# Hard-wall alpha1=3 purification: 100 Born trajectories

Nx=20, Ny=40, T=160. Perfect correction, no postselection. Every-cycle entropy, spatial contours and spectra, plus final covariances. New output collection; existing alpha1=1 data are untouched. Expected 13–16 A100 hours with ten-cycle resume.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import torch

REPORT_ONLY = False
MAX_NEW_EXECUTION_BATCHES = None  # set 1 to limit this session
BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/20_hard_wall_alpha3_purification')
LOCAL_BUNDLE_DIR = Path('/content/20_hard_wall_alpha3_purification')
SCRATCH_ROOT = Path('/content/hard_wall_alpha3_purification_scratch')
# Locked scientific contract; resume/session controls are above.
CONFIG = {'sampling_revision': 'hard_wall_alpha3_maxmix_nx20_ny40_s100_4ny_v1',
 'root_seed': 2026092403,
 'Nx': 20,
 'Ny_values': [40],
 'samples_per_case': 100,
 'cycles_multiplier': 4,
 'nshell': 1,
 'alpha_1': 3.0,
 'alpha_2': 30.0,
 'trial_orbitals': 'X',
 'filling_fraction': 0.5,
 'n_a': 0.5,
 'init_mode': 'maxmix',
 'sequence': 'raster_y',
 'perfect_correction': True,
 'postselect': False,
 'dtype': 'complex128',
 'device': 'cuda:0',
 'backend': 'local',
 'state_representation': 'covariance',
 'result_shard_size': 5,
 'segment_cycles': 10,
 'gpu_memory_budget_bytes': 38000000000,
 'gpu_allocator_limit_bytes': 35000000000,
 'resume_initial_purity_tolerance': 0.50000001,
 'execution_batch_size_by_Ny': {'40': 40},
 'observer_sample_chunk_by_Ny': {'40': 5},
 'constructions': {'hard': {'DW': True,
                            'dw_truncation': True,
                            'meas_slab_only': True}},
 'observables': {'cycles': '0..4Ny inclusive',
                 'occupation_spectrum': True,
                 'cell_entropy_contour': True,
                 'cell_charge_variance_contour': True,
                 'final_centered_covariance': True,
                 'measurement_log_probability': True,
                 'cumulative_log_probability': True,
                 'log_probability_dtype': 'float64',
                 'hard_log_probability_origin': 'after_born_conditioned_exterior_preparation',
                 'soft_log_probability_origin': 'global_maxmix_cycle_zero',
                 'entropy_log_base': 'natural'}}
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs') / CONFIG['sampling_revision']
if not REPORT_ONLY:
    if not torch.cuda.is_available():
        raise RuntimeError('An A100 GPU is required.')
    name = torch.cuda.get_device_name(0)
    if 'A100' not in name or torch.cuda.get_device_properties(0).total_memory / 1024**3 < 35:
        raise RuntimeError(f'Expected A100 40-GB-class runtime, got {name}')
    print('[GPU]', name, '; dtype:', torch.complex128, flush=True)
print(json.dumps({'config': CONFIG, 'bundle': str(BUNDLE_DRIVE_DIR), 'output': str(OUTPUT_ROOT)}, indent=2), flush=True)


## Stage locally and run/resume

Native outer shard and inner cycle bars. An interruption resumes the last verified ten-cycle checkpoint. Use a fresh runtime if another campaign has already imported its engine.


In [ ]:
import importlib.util
import shutil
import sys

if not BUNDLE_DRIVE_DIR.is_dir():
    raise FileNotFoundError(f'Missing uploaded folder: {BUNDLE_DRIVE_DIR}')
LOCAL_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
for relative in ['run_campaign.py', 'purification_observer.py', 'src/classA_U1FGTN_gpu.py', 'src/occupied_frame_gpu.py']:
    destination = LOCAL_BUNDLE_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BUNDLE_DRIVE_DIR / relative, destination)
for name in ['classA_U1FGTN_gpu', 'occupied_frame_gpu', 'purification_observer']:
    loaded = sys.modules.get(name)
    if loaded is not None and not Path(loaded.__file__).resolve().is_relative_to(LOCAL_BUNDLE_DIR):
        raise RuntimeError(f'{name} was imported from another campaign. Restart the runtime first.')
config_path = LOCAL_BUNDLE_DIR / 'resolved_config.json'
config_path.write_text(json.dumps(CONFIG, indent=2) + '\n')
sys.path.insert(0, str(LOCAL_BUNDLE_DIR))
spec = importlib.util.spec_from_file_location('_alpha3_purification_runner', LOCAL_BUNDLE_DIR / 'run_campaign.py')
runner = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = runner
spec.loader.exec_module(runner)
args = ['--config', str(config_path), '--output-root', str(OUTPUT_ROOT), '--scratch-root', str(SCRATCH_ROOT)]
if REPORT_ONLY:
    args.append('--report-only')
if MAX_NEW_EXECUTION_BATCHES is not None:
    args.extend(['--max-new-execution-batches', str(int(MAX_NEW_EXECUTION_BATCHES))])
print('[launch in notebook kernel]', runner.__file__, flush=True)
runner.main(args)
print('[notebook] requested queue completed successfully', flush=True)


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
